# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# --- Install (if needed) ---
!pip install huggingface_hub pandas pyarrow -q

# --- Auth via Colab Secrets ---
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import pandas as pd
import numpy as np

# --- Load March 2026 partition + dimension tables ---
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)
dim_clients = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet")
dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")

# --- Rebuild working df: join + filters (same as ML-05/ML-06) ---
df = df_march.copy()
df = df.merge(
    dim_content[['content_hash_id', 'content_type', 'word_count', 'content_created_date']],
    on='content_hash_id', how='left'
)
df = df[df['gsc_data_available'] == True]
df = df[df['gsc_impressions'] >= 500]

df['content_age_days'] = (
    pd.to_datetime(df['report_date']) - pd.to_datetime(df['content_created_date'])
).dt.days
df['ctr_check'] = df['gsc_clicks'] / df['gsc_impressions']

print(df.shape)


(101451, 36)


## Section 1: My Rule and Its Reason Codes

**The rule, in plain words:**
Flag a piece of content as an action opportunity if it's getting real traffic exposure
(high impressions) but ranks poorly (worst position quartile) — because Test 2 in ML-06
CONFIRMED that poor position is a strong, reliable driver of low CTR. This is a lever we
know works: fix the ranking, and CTR should follow. Content with low impressions isn't
worth the effort regardless of position (not enough exposure to matter); content that
already ranks well doesn't need this particular fix.

**Thresholds (grounded in ML-06's quartile splits):**
- High impressions = top 25% of impressions (`gsc_impressions` ≥ 75th percentile)
- Poor position = worst 25% of position (`gsc_avg_position` in worst quartile — higher
  number = worse rank)

**Reason codes (the rule outputs exactly one per row):**
- `HIGH_VOL_POOR_RANK` — high impressions AND worst-quartile position → the core opportunity signal
- `HIGH_VOL_OK_RANK` — high impressions but position is NOT in the worst quartile → already decent, lower priority
- `LOW_VOL` — impressions below the top quartile → not enough exposure to prioritize, regardless of position

**Action labels (one per reason code):**
- `HIGH_VOL_POOR_RANK` → `FIX_RANKING` (highest priority — proven lever, real exposure to capture)
- `HIGH_VOL_OK_RANK` → `MONITOR` (has exposure and decent rank already; watch for drift)
- `LOW_VOL` → `DEPRIORITIZE` (not enough impression volume for this fix to matter yet)

**Score:** a simple composite — impression rank (0-1) × position-badness rank (0-1) — so
content that is BOTH high volume AND poorly ranked scores highest, and either dimension
being weak pulls the score down. This keeps the score honest to the single CONFIRMED signal
rather than inventing a black-box formula.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import numpy as np
import os

# --- Build the ranked queue per the rule defined in Section 1 ---

wq = df.copy()

# Impression percentile rank (within full dataset — this is a global rule)
wq['impr_pct'] = wq['gsc_impressions'].rank(pct=True)

# Position "badness" — higher position number = worse rank, so badness rank = pct rank of position
wq['position_badness_pct'] = wq['gsc_avg_position'].rank(pct=True)

# --- Assign reason code per the plain-word rule ---
def assign_reason(row):
    high_vol = row['impr_pct'] >= 0.75
    poor_rank = row['position_badness_pct'] >= 0.75
    if high_vol and poor_rank:
        return 'HIGH_VOL_POOR_RANK'
    elif high_vol and not poor_rank:
        return 'HIGH_VOL_OK_RANK'
    else:
        return 'LOW_VOL'

wq['reason_code'] = wq.apply(assign_reason, axis=1)

# --- Map reason code to action label ---
action_map = {
    'HIGH_VOL_POOR_RANK': 'FIX_RANKING',
    'HIGH_VOL_OK_RANK': 'MONITOR',
    'LOW_VOL': 'DEPRIORITIZE',
}
wq['action'] = wq['reason_code'].map(action_map)

# --- Score: impression rank x position-badness rank (both 0-1, so score is 0-1) ---
wq['score'] = wq['impr_pct'] * wq['position_badness_pct']

# --- Build the daily-grain queue first ---
queue_cols = ['content_hash_id', 'client_hash_id', 'report_date',
              'score', 'reason_code', 'action',
              'gsc_impressions', 'gsc_avg_position']

ranked_queue = wq[queue_cols].sort_values('score', ascending=False).reset_index(drop=True)

print("=== Daily-grain reason code counts ===")
print(ranked_queue['reason_code'].value_counts())
print(f"Total daily rows: {len(ranked_queue)}")

# --- Collapse to one row per content_hash_id (average across its days in March) ---
queue_dedup = (
    ranked_queue.groupby('content_hash_id')
    .agg(
        client_hash_id=('client_hash_id', 'first'),
        avg_score=('score', 'mean'),
        reason_code=('reason_code', lambda x: x.mode()[0]),  # most common reason code across days
        action=('action', lambda x: x.mode()[0]),
        avg_impressions=('gsc_impressions', 'mean'),
        avg_position=('gsc_avg_position', 'mean'),
        days_seen=('report_date', 'count'),
    )
    .reset_index()
    .sort_values('avg_score', ascending=False)
    .reset_index(drop=True)
)

print(f"\n=== Deduplicated content-level queue ===")
print(f"Unique content pieces: {len(queue_dedup)}")
print(f"Unique clients represented: {queue_dedup['client_hash_id'].nunique()}")
print(f"\nAction label counts:\n{queue_dedup['action'].value_counts()}")
print(f"\nTop 20:")
print(queue_dedup.head(20))

# --- Write the final CSV (content-level, deduplicated) ---
os.makedirs('work/outputs', exist_ok=True)
queue_dedup.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("\nWritten to work/outputs/baseline_action_score.csv")

=== Daily-grain reason code counts ===
reason_code
LOW_VOL               76076
HIGH_VOL_OK_RANK      18822
HIGH_VOL_POOR_RANK     6553
Name: count, dtype: int64
Total daily rows: 101451

=== Deduplicated content-level queue ===
Unique content pieces: 9214
Unique clients represented: 26

Action label counts:
action
DEPRIORITIZE    8276
MONITOR          765
FIX_RANKING      173
Name: count, dtype: int64

Top 20:
             content_hash_id           client_hash_id  avg_score  \
0   content_ab91e088440ace78  client_23a62021009f63c4   0.921425   
1   content_73aa61dcedebbf30  client_23a62021009f63c4   0.908398   
2   content_96e6613b42b52c42  client_23a62021009f63c4   0.891016   
3   content_36e53e9c707674fc  client_23a62021009f63c4   0.887491   
4   content_559cdd76da9306de  client_23a62021009f63c4   0.881688   
5   content_fa84f5976d5fe3c1  client_23a62021009f63c4   0.872045   
6   content_c367b0ca57f3559b  client_23a62021009f63c4   0.866146   
7   content_a11bd5663919f057  client_20259

In [4]:
# Does one client just have way more content overall, or way more HIGH_VOL_POOR_RANK specifically?
print("Content count by client (top 10):")
print(queue_dedup['client_hash_id'].value_counts().head(10))

print("\nFIX_RANKING count by client (top 10):")
print(queue_dedup[queue_dedup['action'] == 'FIX_RANKING']['client_hash_id'].value_counts().head(10))

Content count by client (top 10):
client_hash_id
client_73cda7b4e4f265ea    2595
client_23a62021009f63c4    2088
client_62f4a7e64f5e0096    1837
client_20259bd6705d81d4     750
client_e547b89c05043229     677
client_e5c2aa26a8598242     332
client_fef1a8f436438636     331
client_a80fca3f171ed1de     180
client_08a6a72ff48e62c0     137
client_0fa64a184f18a4a0      58
Name: count, dtype: int64

FIX_RANKING count by client (top 10):
client_hash_id
client_23a62021009f63c4    129
client_20259bd6705d81d4     33
client_fef1a8f436438636      6
client_e5c2aa26a8598242      3
client_e547b89c05043229      2
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Section 3: Top-20 Review

All 20 top-ranked pieces carry reason_code HIGH_VOL_POOR_RANK / action FIX_RANKING — they all
sit above the 75th percentile on both impression volume and position badness. Average impressions
range ~1,865–6,277; average position ranges ~30.8–47.0, consistent with the rule's design.

**Confidence note (applies to all 20):** Medium-low. The ranking is directionally correct (all
20 genuinely have real exposure AND poor rank) but the *ordering* is likely skewed by which
client they belong to — see Section 4.

**What would make each wrong:** any of these could be a false positive if the content is already
scheduled for retirement, seasonal (naturally declining exposure, not fixable by ranking work),
or if position is poor because of an unfixable structural issue (e.g., a new client whose whole
site hasn't been indexed long) rather than a fixable on-page problem.

17 of the top 20 belong to a single client (client_23a62021009f63c4). This is flagged and
investigated in Section 4 rather than treated as 20 independent findings.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Section 4: Weak Picks + Leakage Check

**Weak pattern found:** 129 of the total 173 FIX_RANKING flags (75%) belong to a single client
(client_23a62021009f63c4), despite that client not having the most overall content (2,088 vs.
2,595 for the top client by volume). Meanwhile client_73cda7b4e4f265ea — the client with the
MOST content overall — has zero FIX_RANKING flags.

**Why this happens:** the rule ranks impressions and position globally (across all clients),
not per-client. A client with systemically higher raw traffic will clear the "top 25%
impressions" bar far more often than a lower-traffic client, even if that lower-traffic client
has content that's *relatively* underperforming for its own scale. The rule is currently
measuring "which client has the most raw traffic," not "which content underperforms relative
to its own peers" — closer to the within-client design we used for the ML-05 label, which this
rule did not replicate.

**What would fix it:** re-rank impr_pct and position_badness_pct within each client_hash_id
(groupby before rank), the same approach used in ML-05, so a low-traffic client's genuinely
poor-performing content can still surface instead of being permanently outranked by one
high-traffic client.

**Leakage check:** No product flags, existing-system scores, or future-window data were used.
impr_pct and position_badness_pct are both computed from the same-day gsc_impressions and
gsc_avg_position — no label-derived or future information is present. This is a design/fairness
weakness (global vs. per-client ranking), not a leakage issue.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.